# Introduction

This notebook presents an implementaiton of the **indexing pipeline for a GraphRAG via Lexical Search**, using:
- Tech Stack: Neo4j and LangChain
- Query Method: Lexical Search via Cypher query language (not vector search/semantic search)

**Setting up**

You will need to have a running Neo4j instance. One option is to create a [free Neo4j database instance in their Aura cloud service](https://neo4j.com/cloud/platform/aura-graph-database/). You can also run the database locally using the [Neo4j Desktop application](https://neo4j.com/download/), or running a docker container. Check the references for more information.

**Sources**
1. [DeepLearning.AI | Knowledge Graphs for RAG](https://learn.deeplearning.ai/courses/knowledge-graphs-rag/lesson/5/constructing-a-knowledge-graph-from-text-documents)
2. [GraphRAG | Lexical Graph](https://graphr.ag/reference/knowledge-graph/lexical-graph/)
3. [GraphRAG | Lexical Graph with Sibling Structure](https://graphr.ag/reference/knowledge-graph/lexical-graph-sibling-structure/)
4. [GraphRAG | Domain Graph](https://graphr.ag/reference/knowledge-graph/domain-graph/)
5. [LangChain | How to construct knowledge graphs](https://python.langchain.com/v0.2/docs/how_to/graph_constructing/)
6. [LangChain | LLMGraphTransformer](https://api.python.langchain.com/en/latest/graph_transformers/langchain_experimental.graph_transformers.llm.LLMGraphTransformer.html)
7. [LangChain | RetrievalQAWithSourcesChain](https://api.python.langchain.com/en/latest/chains/langchain.chains.qa_with_sources.retrieval.RetrievalQAWithSourcesChain.html)

In [1]:
! pip install --quiet \
              langchain==0.2.14 \
              langchain-openai==0.1.20 \
              langchain-community==0.2.12 \
              langchain-experimental==0.0.64 \
              neo4j-graphrag==1.2.0 \
              pymupdf==1.24.13 \
              pypdf==5.1.0 \
              python-dotenv==1.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.8/997.8 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.6 MB/s eta 0:00:00


In [70]:
import asyncio
import hashlib
import os
import requests
import textwrap

from datetime import datetime
from dotenv import load_dotenv, find_dotenv
from langchain.chains import GraphCypherQAChain
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.document_loaders import PyMuPDFLoader
from langchain.prompts.prompt import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from neo4j import GraphDatabase
#from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from tqdm import tqdm

In [3]:
use_google_drive_api = True
google_drive_path = 'gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain'

if(use_google_drive_api):
  from google.colab import drive
  drive.mount('/content/gdrive/', force_remount=True)
  # Changing the current working directory to the notebook folder
  os.chdir(google_drive_path)
  print(os.getcwd())

Mounted at /content/gdrive/
/content/gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain


In [71]:
# Enverionment variables
_ = load_dotenv(find_dotenv(), override=True) # read local .env file

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_ENDPOINT = os.getenv('OPENAI_BASE_URL') + '/embeddings'
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# Global constants
VECTOR_INDEX_NAME = "vector_index"
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'content'
VECTOR_EMBEDDING_PROPERTY = 'embedding'

# 0. Create the dataset

In [72]:
# List of URLs to download
pdf_urls = [
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-1.pdf",
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-2.pdf",
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-3.pdf",
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-4.pdf"
]

# Directory to save PDFs
data_dir = "data"

In [73]:
# Create the directory to save PDFs if it doesn't exist
os.makedirs(data_dir, exist_ok=True)  #

# Headers to mimic a browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36"
}

# Download each PDF
for url in pdf_urls:
    # Extract file name from URL
    file_name = os.path.join(data_dir, url.split("/")[-1])

    try:
        # Download the file and save it
        response = requests.get(url, headers=headers, timeout=10)  # Set a timeout for reliability
        response.raise_for_status()  # Check if the request was successful
        with open(file_name, "wb") as file:
            file.write(response.content)
        print(f"Downloaded: {file_name}")
    except requests.exceptions.RequestException as e:
        print(f"Failed to download {url}. Error: {e}")

Downloaded: data/Goal-1.pdf
Downloaded: data/Goal-2.pdf
Downloaded: data/Goal-3.pdf
Downloaded: data/Goal-4.pdf


# 1. Load and Split the PDF files

In [74]:
# Instantiate hash function

def generate_hash(data: str) -> str:
    """
    Generates a SHA-256 hash for the given data.

    Parameters:
    - data (str): The input data to be hashed.

    Returns:
    - str: The hexadecimal hash string.
    """
    # Encode the data to bytes, required by hashlib
    data_bytes = data.encode('utf-8')

    # Create a SHA-256 hash object
    sha256_hash = hashlib.sha256()

    # Update the hash object with the data
    sha256_hash.update(data_bytes)

    # Get the hexadecimal representation of the hash
    return sha256_hash.hexdigest()

# Example usage
input_data = "Hello, World!"
hash_result = generate_hash(input_data)
print(f"The SHA-256 hash of '{input_data}' is: {hash_result}")

The SHA-256 hash of 'Hello, World!' is: dffd6021bb2bd5b0af676290809ec3a53191dd81c7f70a4b28688a362182986f


In [75]:
# Initialize the text splitter with a max chunk size of 1000 characters and overlap of 200 characters
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Function to load and split PDFs
def load_and_split_pdfs(directory):
    documents = []
    chunks = []

    file_names = os.listdir(directory)
    for file_name in tqdm(file_names):
        if file_name.endswith(".pdf"):
            # Create a doc ID
            doc_id = generate_hash(file_name)

            # Load the PDF
            file_path = os.path.join(directory, file_name)
            loader = PyMuPDFLoader(file_path)
            pages = loader.load()

            chunk_index = 0
            chunks_with_metadata = []
            # Split the loaded text into chunks
            for page in pages:
                split_pages = text_splitter.split_documents([page])

                # Create chunk with metadata
                for split_page in split_pages:
                    content = split_page.page_content

                    # Create a chunk ID
                    chunk_id = generate_hash(content)

                    chunk_with_metadata = {
                        'content': content,
                        'chunkId': chunk_id,
                        'chunkIndex': chunk_index,
                        'page': split_page.metadata['page'],
                        'source': split_pages[0].metadata['source'],
                        'docId': doc_id,
                        'createdAt': datetime.now().isoformat()

                    }
                    chunks_with_metadata.append(chunk_with_metadata)

                    chunk_index += 1

            chunks.extend(chunks_with_metadata)

            # Create file with metadata
            doc_with_metadata = {
                'docId': doc_id,
                'docName': file_name,
                'source': split_pages[0].metadata['source'],
                'totalPages': split_pages[0].metadata['total_pages'],
                'totalChunks': len(chunks_with_metadata),
                'createdAt': datetime.now().isoformat()

            }

            documents.append(doc_with_metadata)


    return chunks, documents

# Load and split the documents
chunks_with_metadata, docs_with_metadata = load_and_split_pdfs(data_dir)

100%|██████████| 4/4 [00:00<00:00, 30.49it/s]


In [76]:
len(docs_with_metadata)

4

In [77]:
docs_with_metadata[0]

{'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43',
 'docName': 'Goal-1.pdf',
 'source': 'data/Goal-1.pdf',
 'totalPages': 2,
 'totalChunks': 6,
 'createdAt': '2024-11-26T17:44:18.639193'}

In [78]:
len(chunks_with_metadata)

19

In [79]:
chunks_with_metadata[0]

{'content': 'What’s the goal here?\nTo end poverty in all its \nforms everywhere by 2030.\nWhy?\nMore than 700 million peo\xad\nple, or 11% of the world pop\xad\nulation, still live in extreme \npoverty and is struggling to \nfulfil the most basic needs \nlike health, education, and \naccess to water and san\xad\nitation, to name a few.. \nYes. The overwhelming \nmajority of people living \non less than $1.90 a day \nlive in Southern Asia and \nsub-Saharan Africa and \nthey account for about 70 \nper cent of the global total \nof extremely poor people.\nIn 2017, economic losses \nattributed to natu\xad\nral disasters were esti\xad\nmated at over $300 \nbillion, among the highest \nlosses in recent years.\nHowever, this issue also \naffects developed coun\xad\ntries. Right now there \nare 30 million children \ngrowing up poor in the \nworld’s richest countries.\nBy 2030  \n167 million  \nchildren \nwill live in \nextreme \npoverty\nif the world\ndoesn’t take\n action \nto improve \nheal

In [80]:
chunks_with_metadata[1]

{'content': 'world’s richest countries.\nBy 2030  \n167 million  \nchildren \nwill live in \nextreme \npoverty\nif the world\ndoesn’t take\n action \nto improve \nhealth \nand education\nNO POVERTY: \nWHY IT MATTERS\n© UN Photo / Eskinder Debebe',
 'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b',
 'chunkIndex': 1,
 'page': 0,
 'source': 'data/Goal-1.pdf',
 'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43',
 'createdAt': '2024-11-26T17:44:18.638472'}

In [81]:
chunks_with_metadata[2]

{'content': 'Why is there so much \npoverty in the world?\nPoverty has many dimen\xad\nsions, but its causes \ninclude unemploy\xad\nment, social exclusion, \nand high vulnerability \nof certain population \nto disasters, diseases \nand other phenomena \nwhich prevent them \nfrom being productive.\nI’m not poor. Why should I \ncare about other people’s \neconomic situation?\nThere are many reasons, \nbut in short, because as \nhuman beings, our well-\nbeing is linked to each \nother. Growing inequality \nis detrimental to economic \ngrowth and undermines \nsocial cohesion, increas\xad\ning political and social \ntensions and, in some \ncircumstances, driving \ninstability and conflicts.\nCan we actually \nachieve this goal?\nYes. To end extreme pov\xad\nerty worldwide in 20 \nyears, economist Jeffrey \nSachs calculated that the \ntotal cost per year would \nbe about $175 billion. This \nrepresents less than one \npercent of the combined \nincome of the richest \ncountries in the world.

In [82]:
chunks_with_metadata[6]

{'content': 'What’s the goal here?\nTo end hunger, achieve \nfood security and improved \nnutrition and promote \nsustainable agriculture\nWhy?\nExtreme hunger and mal\xad\nnutrition remains a barrier \nto sustainable develop\xad\nment and creates a trap \nfrom which people cannot \neasily escape. Hunger and \nmalnutrition mean less \nproductive individuals, who \nare more prone to disease \nand thus often unable to \nearn more and improve \ntheir livelihoods. There are \nnearly 800 million people \nwho suffer from hunger \nworldwide, the vast major\xad\nity in developing countries.   \nA profound \nchange of the \nglobal food \nand agriculture \nsystem is needed \nto nourish \ntoday’s 795 \nmillion \nhungry + \nthe additional \n2 billion \nincrease in global \npopulation \nexpected by \n2050\nZERO HUNGER: \nWHY IT MATTERS\n© UN Photo / Albert González Farran',
 'chunkId': '40eb9458bea528ae66e8e69369566e7a81c0952283d130b0836bcfa123ee71eb',
 'chunkIndex': 0,
 'page': 0,
 'source': 'data

# 2. Build the Knowledge Graph

In [83]:
kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

## 2.1. Lexical Graph

### Create indexes

In [84]:
# Create a uniqueness constraint to avoid duplicate chunks
kg.query("""
CREATE CONSTRAINT unique_chunk IF NOT EXISTS
    FOR (c:Chunk) REQUIRE c.chunkId IS UNIQUE
""")


[]

In [85]:
# Create a uniqueness constraint to avoid duplicate documents
kg.query("""
CREATE CONSTRAINT unique_document IF NOT EXISTS
    FOR (d:Document) REQUIRE d.docId IS UNIQUE
""")


[]

In [86]:
# Create vector index
kg.query("""
         CREATE VECTOR INDEX `""" + VECTOR_INDEX_NAME + """` IF NOT EXISTS
          FOR (c:Chunk) ON (c.embedding)
          OPTIONS { indexConfig: {
            `vector.dimensions`: 1536,
            `vector.similarity_function`: 'cosine'
         }}
""")

[]

In [87]:
kg.query("SHOW INDEXES")

[{'id': 0,
  'name': 'index_343aff4e',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 1,
  'name': 'index_f7700477',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 2,
  'name': 'unique_chunk',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['chunkId'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'unique_chunk',
  'lastRead': None,
  'readCount': None},
 {'id': 4,
  'name': 'unique_document',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'ent

### Create graph nodes using text chunks

In [88]:
merge_chunk_node_query = """
MERGE(mergedChunk:Chunk {chunkId: $chunkParam.chunkId})
    ON CREATE SET
        mergedChunk.content = $chunkParam.content,
        mergedChunk.chunkIndex = $chunkParam.chunkIndex,
        mergedChunk.page = $chunkParam.page,
        mergedChunk.source = $chunkParam.source,
        mergedChunk.docId = $chunkParam.docId,
        mergedChunk.createdAt = $chunkParam.createdAt
RETURN mergedChunk
"""

for chunk in tqdm(chunks_with_metadata):
    kg.query(merge_chunk_node_query, params={'chunkParam': chunk})

100%|██████████| 19/19 [00:01<00:00,  9.77it/s]


In [89]:
kg.query("""
         MATCH (n)
         RETURN count(n) as nodeCount
         """)

[{'nodeCount': 19}]

In [90]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Relationship properties:

The relationships:



### Calculate embedding vectors for chunks and populate index
- This query calculates the embedding vector and stores it as a property called `textEmbedding` on each `Chunk` node.

In [91]:
# https://platform.openai.com/docs/api-reference/embeddings/create

kg.query("""
    MATCH (chunk:Chunk) WHERE chunk.embedding IS NULL
    WITH chunk, genai.vector.encode(
      chunk.content,
      "OpenAI",
      {
        token: $openAiApiKey,
        endpoint: $openAiEndpoint
      }) AS vector
    CALL db.create.setNodeVectorProperty(chunk, "embedding", vector)
    """,
    params={"openAiApiKey":OPENAI_API_KEY, "openAiEndpoint": OPENAI_ENDPOINT} )

[]

In [92]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, embedding: LIST, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Relationship properties:

The relationships:



### Create graph nodes using documents

In [93]:
merge_document_node_query = """
    MERGE (d:Document {docId: $docParam.docId })
      ON CREATE
        SET d.docName = $docParam.docName
        SET d.source = $docParam.source
        SET d.totalPages = $docParam.totalPages
        SET d.totalChunks = $docParam.totalChunks
        SET d.createdAt = $docParam.createdAt
"""

for doc in tqdm(docs_with_metadata):
    kg.query(merge_document_node_query, params={'docParam': doc})

100%|██████████| 4/4 [00:00<00:00,  5.55it/s]


In [94]:
kg.query("MATCH (d:Document) RETURN count(d) as documentCount")

[{'documentCount': 4}]

In [95]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, embedding: LIST, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Document {docId: STRING, source: STRING, createdAt: STRING, docName: STRING, totalPages: INTEGER, totalChunks: INTEGER}
Relationship properties:

The relationships:



### Create graph relationships between chunks

#### Add a NEXT relationship between subsequent chunks
- Use the `apoc.nodes.link` function from Neo4j to link ordered list of `Chunk` nodes with a `NEXT_CHUNK` relationship

In [96]:
create_next_relationship_query = """
  MATCH (c:Chunk)
  WHERE c.docId = $docParam.docId
  WITH c
    ORDER BY c.chunkIndex ASC
  WITH collect(c) as sorted_chunk_list
    CALL apoc.nodes.link(
        sorted_chunk_list,
        "NEXT_CHUNK",
        {avoidDuplicates: true}
    )
  RETURN size(sorted_chunk_list) as number_of_chunks
"""

for doc in tqdm(docs_with_metadata):
    results = kg.query(create_next_relationship_query, params={'docParam': doc})
    docName = doc['docName']
    number_of_chunks = results[0]['number_of_chunks']
    print(f"{docName}: {number_of_chunks} chunk - {number_of_chunks-1} Links")

 50%|█████     | 2/4 [00:00<00:00,  5.45it/s]

Goal-1.pdf: 6 chunk - 5 Links
Goal-2.pdf: 4 chunk - 3 Links


100%|██████████| 4/4 [00:00<00:00,  5.85it/s]

Goal-3.pdf: 5 chunk - 4 Links
Goal-4.pdf: 4 chunk - 3 Links


In [97]:
query = """
MATCH (:Chunk)-[r:NEXT_CHUNK]->(:Chunk)
RETURN count(r) as number_of_next_relationships
"""

kg.query(query)

[{'number_of_next_relationships': 15}]

In [98]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, embedding: LIST, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Document {docId: STRING, source: STRING, createdAt: STRING, docName: STRING, totalPages: INTEGER, totalChunks: INTEGER}
Relationship properties:

The relationships:
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)


### Create graph relationships between chunks and documents

#### Add PART_OF relationship between chunks to their document

In [99]:
create_partOf_relationship_query = """
  MATCH (c:Chunk), (d:Document)
    WHERE c.docId = d.docId
  MERGE (c)-[newRelationship:PART_OF]->(d)
  RETURN count(newRelationship)
"""

kg.query(create_partOf_relationship_query)

[{'count(newRelationship)': 19}]

In [100]:
query = """
MATCH (:Chunk)-[r:PART_OF]->(:Document)
RETURN count(r) as number_of_partOf_relationships
"""

kg.query(query)

[{'number_of_partOf_relationships': 19}]

In [101]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, embedding: LIST, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Document {docId: STRING, source: STRING, createdAt: STRING, docName: STRING, totalPages: INTEGER, totalChunks: INTEGER}
Relationship properties:

The relationships:
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
(:Chunk)-[:PART_OF]->(:Document)


#### Add FIRST_CHUNK relationship between documents and their first chunk

In [102]:
create_firstChunk_relationship_query = """
  MATCH (c:Chunk), (d:Document)
  WHERE c.docId = d.docId
    AND c.chunkIndex = 0
  WITH c, d
    MERGE (d)-[r:FIRST_CHUNK {docName: d.docName}]->(c)
  RETURN count(r)
"""

kg.query(create_firstChunk_relationship_query)

[{'count(r)': 4}]

In [103]:
query = """
MATCH (:Document)-[r:FIRST_CHUNK]->(:Chunk)
RETURN count(r) as number_of_firstChunk_relationships
"""

kg.query(query)

[{'number_of_firstChunk_relationships': 4}]

In [104]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, embedding: LIST, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Document {docId: STRING, source: STRING, createdAt: STRING, docName: STRING, totalPages: INTEGER, totalChunks: INTEGER}
Relationship properties:
FIRST_CHUNK {docName: STRING}
The relationships:
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
(:Chunk)-[:PART_OF]->(:Document)
(:Document)-[:FIRST_CHUNK]->(:Chunk)


## 2.2. Domain Graph

- Use LLMGraphTransformer from LangChain to identity entities and relationships between them for each chunk
- Use cypher to manually create the relationships between chunks and their respective entities

In [40]:
# Instantiate an LLM

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini",
    temperature=0,
    seed=0
)

### Create entity nodes
- Use LLMGraphTransformer from LangChain to identity entities and relationships between them

In [41]:
# The LLMGraphTransformer converts text documents into structured graph documents
# by leveraging a LLM to parse and categorize entities and their relationships.

allowed_nodes=["Person", "Country", "Organization"]
allowed_relationships=[]
node_properties=['description']
relationship_properties=['description']

llm_transformer = LLMGraphTransformer(
  llm=llm,
  allowed_nodes=allowed_nodes,
  allowed_relationships=allowed_relationships,
  node_properties=node_properties,
  relationship_properties=relationship_properties,
  strict_mode=True
)

In [42]:
from langchain.schema import Document

doc_chunks = {}
for doc in docs_with_metadata:
  docName = doc['docName']
  chunks = []
  for chunk in chunks_with_metadata:
    metadata = {k:v for k, v in chunk.items() if k!='content'}
    document = Document(page_content=chunk['content'], metadata=metadata)
    chunks.append(document)
  doc_chunks[docName] = chunks

In [43]:
doc_chunks[list(doc_chunks.keys())[0]][0]

Document(metadata={'chunkId': '26beb166144aec72d804302fd4ca9359360f681734fb9213b1e77746537a5ca7', 'chunkIndex': 0, 'page': 0, 'source': 'data/Goal-1.pdf', 'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43', 'createdAt': '2024-11-26T16:49:45.134280'}, page_content='What’s the goal here?\nTo end poverty in all its \nforms everywhere by 2030.\nWhy?\nMore than 700 million peo\xad\nple, or 11% of the world pop\xad\nulation, still live in extreme \npoverty and is struggling to \nfulfil the most basic needs \nlike health, education, and \naccess to water and san\xad\nitation, to name a few.. \nYes. The overwhelming \nmajority of people living \non less than $1.90 a day \nlive in Southern Asia and \nsub-Saharan Africa and \nthey account for about 70 \nper cent of the global total \nof extremely poor people.\nIn 2017, economic losses \nattributed to natu\xad\nral disasters were esti\xad\nmated at over $300 \nbillion, among the highest \nlosses in recent years.\nHowever,

In [44]:
graph_documents = {}
for doc_name, chunks in tqdm(doc_chunks.items()):
  graph_documents[doc_name] = llm_transformer.convert_to_graph_documents(chunks)

for doc_name in doc_chunks.keys():
  for chunk_index in range(len(doc_chunks[doc_name])):
    metadata = graph_documents[doc_name][chunk_index].source.metadata
    for node in graph_documents[doc_name][chunk_index].nodes:
      if not node.properties.get('description', None):
        graph_documents[doc_name][chunk_index].nodes.remove(node)
      else:
        node.properties['label'] = node.type
        node.type = 'Entity'
        node.properties['docId'] = metadata['docId']
        node.properties['chunkId'] = metadata['chunkId']
    for relationship in graph_documents[doc_name][chunk_index].relationships:
      if not relationship.properties.get('description', None):
        graph_documents[doc_name][chunk_index].relationships.remove(relationship)
      else:
        relationship.source.type = 'Entity'
        relationship.target.type = 'Entity'
        relationship.properties['label'] = relationship.type
        relationship.type = 'RELATES_TO'

100%|██████████| 4/4 [06:44<00:00, 101.11s/it]


In [106]:
# Print all nodes and chunks
for doc_name in doc_chunks.keys():
  print('Doc:', doc_name)
  for chunk_index in range(len(doc_chunks[doc_name])):
    print('\tChunk Index:', chunk_index)
    print('\t\tNodes:')
    for node in graph_documents[doc_name][chunk_index].nodes:
      print('\t\t\t', node)
    print('\t\tRelationships:')
    for relationship in graph_documents[doc_name][chunk_index].relationships:
      print('\t\t\t', relationship)

Doc: Goal-1.pdf
	Chunk Index: 0
		Nodes:
		Relationships:
	Chunk Index: 1
		Nodes:
			 id='Un Photo / Eskinder Debebe' type='Entity' properties={'description': 'Photographer for the United Nations', 'label': 'Organization', 'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43', 'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b'}
			 id='World' type='Entity' properties={'description': 'Global context for poverty and education issues', 'label': 'Country', 'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43', 'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b'}
			 id='2030' type='Entity' properties={'description': 'Future year referenced for poverty statistics', 'label': 'Organization', 'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43', 'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b'}
			 id='167 Million' type='Entity' properties={'desc

In [107]:
for doc_name in doc_chunks.keys():
  kg.add_graph_documents(graph_documents[doc_name])

In [108]:
query = """
MATCH (e:Entity)
RETURN count(e) as numberOfEntities
"""

kg.query(query)

[{'numberOfEntities': 85}]

In [109]:
query = """
MATCH (e:Entity)
LIMIT 3
RETURN e
"""

kg.query(query)

[{'e': {'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43',
   'description': 'Photographer for the United Nations',
   'id': 'Un Photo / Eskinder Debebe',
   'label': 'Organization',
   'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b'}},
 {'e': {'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43',
   'description': 'Global context for poverty and education issues',
   'id': 'World',
   'label': 'Country',
   'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b'}},
 {'e': {'docId': '9f0d5d7db93b118a53f9c677b3e95b86491fd4fae73f02084a05a9e601e71e43',
   'description': 'Future year referenced for poverty statistics',
   'id': '2030',
   'label': 'Organization',
   'chunkId': '0e67a287a399fcdb415e6ad5fdd81dcf26986b39b818b597d8784b45f9bf3b0b'}}]

In [110]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Chunk {chunkId: STRING, docId: STRING, embedding: LIST, content: STRING, chunkIndex: INTEGER, page: INTEGER, source: STRING, createdAt: STRING}
Document {docId: STRING, source: STRING, createdAt: STRING, docName: STRING, totalPages: INTEGER, totalChunks: INTEGER}
Entity {chunkId: STRING, docId: STRING, id: STRING, description: STRING, label: STRING}
Relationship properties:
FIRST_CHUNK {docName: STRING}
RELATES_TO {description: STRING, label: STRING}
The relationships:
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
(:Chunk)-[:PART_OF]->(:Document)
(:Document)-[:FIRST_CHUNK]->(:Chunk)
(:Entity)-[:RELATES_TO]->(:Entity)


### Create HAS_ENTITY relationship between entity and chunk nodes
- Use cypher to manually create the relationships between chunks and their respective entities

In [111]:
create_partOf_relationship_query = """
  MATCH (c:Chunk), (e:Entity)
    WHERE c.docId = e.docId
      AND c.chunkId = e.chunkId
  MERGE (c)-[newRelationship:HAS_ENTITY]->(e)
  RETURN count(newRelationship)
"""

kg.query(create_partOf_relationship_query)

[{'count(newRelationship)': 42}]

In [112]:
query = """
MATCH (:Chunk)-[r:HAS_ENTITY]->(:Entity)
RETURN count(r) as number_of_hasEntity_relationships
"""

kg.query(query)

[{'number_of_hasEntity_relationships': 42}]

# 3. Use vector search to find relevant chunks

In [ ]:
def neo4j_vector_search(question, top_k=3):
    """Search for similar nodes using the Neo4j vector index"""
    vector_search_query = """
      WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
      CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
      RETURN node.content AS text, score, node {.source} AS metadata
    """
    similar = kg.query(vector_search_query,
                      params={
                        'question': question,
                        'openAiApiKey':OPENAI_API_KEY,
                        'openAiEndpoint': OPENAI_ENDPOINT,
                        'index_name':VECTOR_INDEX_NAME,
                        'top_k': top_k})
    return similar

In [ ]:
question = 'What is the first Sustainable Development Goal from United Nations?'

search_results = neo4j_vector_search(question)

search_results

[{'text': 'You can also hold your \ngovernment, local leaders \nand other decision-\nmakers accountable to \ntheir commitments to \nimprove people’s access \nto health and health care.  \nTo find out more \nabout Goal #3 and \nthe other Sustainable \nDevelopment Goals, visit: \nhttp://www.un.org/\nsustainabledevelopment',
  'score': 0.918060302734375,
  'metadata': {'source': 'data/Goal-3.pdf'}},
 {'text': 'Access to good health \nand well-being is a human \nright, and that is why the \nSustainable Development \nAgenda offers a new \nchance to ensure that \neveryone can access \nthe highest standards of \nhealth and health care—\nnot just the wealthiest.   \nWhat progress have \nwe made so far?\nMajor progress has been \nmade in several areas, \nincluding in child and \nmaternal health as well as \nin addressing HIV/AIDS, \nmalaria and other diseases. \nMaternal mortality has \nfallen by almost 50 per \ncent since 1990; measles \nvaccines have averted \nnearly 15.6 million deaths \nsin

## 3.1 Customize the results of the similarity search using Cypher

- Extend the vector store definition to accept a Cypher query
- The Cypher query takes the results of the vector similarity search and then modifies them in some way

### 3.1.1 Expand context around a chunk using a window

In [ ]:
def neo4j_vector_search_expanded_window(question, top_k=3):
    """Search for similar nodes using the Neo4j vector index and expand context using a window"""
    vector_search_query = """
      WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
      CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
      MATCH window=
          (:Chunk)-[:NEXT_CHUNK*0..1]->(node)-[:NEXT_CHUNK*0..1]->(:Chunk)
      WITH node, score, window as longestWindow
        ORDER BY length(window) // DESC LIMIT 1
      WITH nodes(longestWindow) as chunkList, node, score
        UNWIND chunkList as chunkRows
      WITH collect(chunkRows.content) as textList, node, score
      RETURN apoc.text.join(textList, " \n ") as text,
          score,
          node {.source} AS metadata
    """
    similar = kg.query(vector_search_query,
                      params={
                        'question': question,
                        'openAiApiKey':OPENAI_API_KEY,
                        'openAiEndpoint': OPENAI_ENDPOINT,
                        'index_name':VECTOR_INDEX_NAME,
                        'top_k': top_k})
    return similar

In [ ]:
question = 'What is the first Sustainable Development Goal from United Nations?'

search_results = neo4j_vector_search_expanded_window(question)

search_results

[{'text': 'You can also hold your \ngovernment, local leaders \nand other decision-\nmakers accountable to \ntheir commitments to \nimprove people’s access \nto health and health care.  \nTo find out more \nabout Goal #3 and \nthe other Sustainable \nDevelopment Goals, visit: \nhttp://www.un.org/\nsustainabledevelopment \n diseases, women will die in \npregnancy and childbirth, \nand health care costs will \ncontinue to plunge millions \nof people into poverty. \nNoncommunicable \ndiseases alone will cost \nlow- and middle-income \ncountries more than $7 \ntrillion in the next 15 years. \nWhat can I do to help?\nYou can start by \npromoting and protecting \nyour own health and \nthe health of those \naround you, by making \nwell-informed choices, \npracticing safe sex and \nvaccinating your children.   \nYou can raise awareness in \nyour community about the \nimportance of good health, \nhealthy lifestyles as well \nas people’s right to quality \nhealth care services.\nTake action thro

### 3.1.2 Enrich context around a chunk using its connected entities

In [ ]:
def neo4j_vector_search_enrich_entities(question, top_k=3):
    """Search for similar nodes using the Neo4j vector index and enrich context with entity info"""
    vector_search_query = """
      WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
      CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
      MATCH (node)-[:HAS_ENTITY]->(e:Entity), (e)-[]->(:Entity)
      WITH collect(
        "(" + e.label + ": " + e.id + ", Description: " + e.description + ")"
      ) as entityInfo, node, score
      RETURN "Context:\n" + node.content + "\nEntities: \n" + apoc.text.join(entityInfo, "\n") as text,
          score,
          node {.source} AS metadata
    """
    similar = kg.query(vector_search_query,
                      params={
                        'question': question,
                        'openAiApiKey':OPENAI_API_KEY,
                        'openAiEndpoint': OPENAI_ENDPOINT,
                        'index_name':VECTOR_INDEX_NAME,
                        'top_k': top_k})
    return similar

In [ ]:
question = 'What is the first Sustainable Development Goal from United Nations?'

search_results = neo4j_vector_search_enrich_entities(question)

search_results

[{'text': 'Context:\nYou can also hold your \ngovernment, local leaders \nand other decision-\nmakers accountable to \ntheir commitments to \nimprove people’s access \nto health and health care.  \nTo find out more \nabout Goal #3 and \nthe other Sustainable \nDevelopment Goals, visit: \nhttp://www.un.org/\nsustainabledevelopment\nEntities: \n',
  'score': 0.918060302734375,
  'metadata': {'source': 'data/Goal-3.pdf'}},
 {'text': 'Context:\nAccess to good health \nand well-being is a human \nright, and that is why the \nSustainable Development \nAgenda offers a new \nchance to ensure that \neveryone can access \nthe highest standards of \nhealth and health care—\nnot just the wealthiest.   \nWhat progress have \nwe made so far?\nMajor progress has been \nmade in several areas, \nincluding in child and \nmaternal health as well as \nin addressing HIV/AIDS, \nmalaria and other diseases. \nMaternal mortality has \nfallen by almost 50 per \ncent since 1990; measles \nvaccines have averted 

In [ ]:
print(search_results[2]['text'])

Context:
contribution of science 
to end poverty has been 
significant. For example, 
it has enabled access 
to safe drinking water, 
reduced deaths caused 
by water-borne diseases, 
and improved hygiene to 
reduce health risks related 
to unsafe drinking water 
and lack of sanitation.
To find out more 
about Goal #1 and 
other Sustainable 
Development Goals visit: 
http://www.un.org/
sustainabledevelopment
Entities: 
(Organization: Science, Description: Contribution of science to end poverty)
(Organization: Safe Drinking Water, Description: Access to safe drinking water)
(Organization: Water-Borne Diseases, Description: Deaths caused by water-borne diseases)
(Organization: Hygiene, Description: Improved hygiene to reduce health risks)
(Organization: Sanitation, Description: Lack of sanitation)
(Organization: Sustainable Development Goals, Description: Goals to end poverty and improve living conditions)


### 3.1.3 Enrich context around a chunk using its entities their relationships

In [ ]:
query = """
MATCH p=(:Entity)-[r]->()
LIMIT 3
RETURN r {.id, .description}
"""

kg.query(query)

[{'r': {'id': None,
   'description': 'if the world doesn’t take action to improve health and education'}},
 {'r': {'id': None,
   'description': 'Projected extreme poverty situation for children'}},
 {'r': {'id': None,
   'description': 'Children living in extreme poverty by 2030'}}]

In [126]:
def neo4j_vector_search_entity_relationships(question, top_k=3):
    """Search for similar nodes using the Neo4j vector index and enrich context with entity info"""
    vector_search_query = """
      WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
      CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
      MATCH (node)-[:HAS_ENTITY]->(e:Entity)-[r:RELATES_TO]->(e2:Entity)
      WITH collect(
        e.id + " (" + e.label + "): " + e.description
      ) as entityInfo,
      collect(
        e.id + " " + r.label + " " + e2.id + " (" + r.description + ")"
      ) as entityRelInfo,
      node, score
      RETURN "Context:\n" + node.content + "\nEntities: \n" + apoc.text.join(entityInfo, "\n") + "\nRelationships: \n" + apoc.text.join(entityRelInfo, "\n") as text,
          score,
          node {.source} AS metadata
    """
    similar = kg.query(vector_search_query,
                      params={
                        'question': question,
                        'openAiApiKey':OPENAI_API_KEY,
                        'openAiEndpoint': OPENAI_ENDPOINT,
                        'index_name':VECTOR_INDEX_NAME,
                        'top_k': top_k})
    return similar

In [127]:
question = 'What is the first Sustainable Development Goal from United Nations?'

search_results = neo4j_vector_search_entity_relationships(question)

search_results

[{'text': 'Context:\ncontribution of science \nto end poverty has been \nsignificant. For example, \nit has enabled access \nto safe drinking water, \nreduced deaths caused \nby water-borne diseases, \nand improved hygiene to \nreduce health risks related \nto unsafe drinking water \nand lack of sanitation.\nTo find out more \nabout Goal #1 and \nother Sustainable \nDevelopment Goals visit: \nhttp://www.un.org/\nsustainabledevelopment\nEntities: \nScience (Organization): Contribution of science to end poverty\nScience (Organization): Contribution of science to end poverty\nScience (Organization): Contribution of science to end poverty\nHygiene (Organization): Improved hygiene to reduce health risks\nGoal #1 (Organization): Sustainable Development Goal #1\nRelationships: \nScience ENABLED Safe Drinking Water (Enabled access to safe drinking water)\nScience REDUCED Water-Borne Diseases (Reduced deaths caused by water-borne diseases)\nScience IMPROVED Hygiene (Improved hygiene to reduce h

In [128]:
print(search_results[0]['text'])

Context:
contribution of science 
to end poverty has been 
significant. For example, 
it has enabled access 
to safe drinking water, 
reduced deaths caused 
by water-borne diseases, 
and improved hygiene to 
reduce health risks related 
to unsafe drinking water 
and lack of sanitation.
To find out more 
about Goal #1 and 
other Sustainable 
Development Goals visit: 
http://www.un.org/
sustainabledevelopment
Entities: 
Science (Organization): Contribution of science to end poverty
Science (Organization): Contribution of science to end poverty
Science (Organization): Contribution of science to end poverty
Hygiene (Organization): Improved hygiene to reduce health risks
Goal #1 (Organization): Sustainable Development Goal #1
Relationships: 
Science ENABLED Safe Drinking Water (Enabled access to safe drinking water)
Science REDUCED Water-Borne Diseases (Reduced deaths caused by water-borne diseases)
Science IMPROVED Hygiene (Improved hygiene to reduce health risks)
Hygiene RELATED_TO Sanita

# 4. Set up a LangChain RAG workflow to chat with the form

In [ ]:
# Instantiate an LLM

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    temperature=0.4,
    model_name="gpt-4o-mini",
)

In [ ]:
# Set up vector store and retriever

neo4j_vector_store = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

retriever = neo4j_vector_store.as_retriever()

In [ ]:
# Instantiate an QA Chain

chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
)

def generate_response_and_display_with_sources(question: str) -> str:
    """Pretty print the chain's response to a question"""
    response = chain.invoke({"question": question}, return_only_outputs=False)
    print(textwrap.fill(response['answer'], 80))
    print("Sources:")
    print(textwrap.fill(response['sources'], 80))

In [ ]:
question = 'What is the first Sustainable Development Goal from United Nations?'

generate_response_and_display_with_sources(question)

The first Sustainable Development Goal from the United Nations is to end poverty
in all its forms everywhere.
Sources:
data/Goal-1.pdf


## 4.1 Customize the results of the similarity search using Cypher

- Extend the vector store definition to accept a Cypher query
- The Cypher query takes the results of the vector similarity search and then modifies them in some way


### 4.1.1 Expand context around a chunk using a window

In [ ]:
retrieval_query_window = """
MATCH window=
    (:Chunk)-[:NEXT_CHUNK*0..1]->(node)-[:NEXT_CHUNK*0..1]->(:Chunk)
WITH node, score, window as longestWindow
  ORDER BY length(window) // DESC LIMIT 1
WITH nodes(longestWindow) as chunkList, node, score
  UNWIND chunkList as chunkRows
WITH collect(chunkRows.content) as textList, node, score
RETURN apoc.text.join(textList, " \n ") as text,
    score,
    node {.source} AS metadata
"""

In [ ]:
# Set up a QA chain that will use the window retrieval query

vector_store_window = Neo4jVector.from_existing_index(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="neo4j",
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=retrieval_query_window, # NEW!!!
)

# Create a retriever from the vector store
retriever_window = vector_store_window.as_retriever()

# Create a chatbot Question & Answer chain from the retriever
chain_window = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever_window
)

def generate_response_windows_and_display_with_sources(question: str) -> str:
    """Pretty print the chain's response to a question"""
    response = chain_window.invoke({"question": question}, return_only_outputs=False)
    print(textwrap.fill(response['answer'], 80))
    print("Sources:")
    print(textwrap.fill(response['sources'], 80))

In [ ]:
question = 'What is the first Sustainable Development Goal from United Nations?'

generate_response_windows_and_display_with_sources(question)

The first Sustainable Development Goal from the United Nations is to end poverty
in all its forms everywhere.
Sources:
data/Goal-1.pdf


### 4.1.2 Enrich context around a chunk using its connected entities

In [ ]:
retrieval_query_entities = """
MATCH (node)-[:HAS_ENTITY]->(e:Entity)
WITH collect(
  "(" + e.label + ": " + e.id + ", Description: " + e.description + ")"
) as entityInfo, node, score
RETURN "Context:\n" + node.content + "\nEntities: \n" + apoc.text.join(entityInfo, "\n") as text,
    score,
    node {.source} AS metadata
"""

In [ ]:
# Set up a QA chain that will use the entity retrieval query

vector_store_entities = Neo4jVector.from_existing_index(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="neo4j",
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=retrieval_query_entities, # NEW!!!
)

# Create a retriever from the vector store
retriever_entities = vector_store_entities.as_retriever()

# Create a chatbot Question & Answer chain from the retriever
chain_entities = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever_entities
)

def generate_response_entities_and_display_with_sources(question: str) -> str:
    """Pretty print the chain's response to a question"""
    response = chain_entities.invoke({"question": question}, return_only_outputs=False)
    print(textwrap.fill(response['answer'], 80))
    print("Sources:")
    print(textwrap.fill(response['sources'], 80))

In [ ]:
question = 'What is the first Sustainable Development Goal from United Nations?'

generate_response_entities_and_display_with_sources(question)

The first Sustainable Development Goal from the United Nations is to end poverty
in all its forms everywhere.
Sources:
data/Goal-1.pdf


### 4.1.3 Enrich context around a chunk using its entities their relationships

In [ ]:
#TBD